# Building Inventory Generator Notebook

This notebook demonstrates how to:
1. Install BrailsPlusPlus for building footprint data extraction
2. Fetch building inventories for a specific location or bounding box
3. Merge with National Structure Inventory (NSI) data for enriched attributes
4. Visualize the results on interactive maps using Folium
5. Export data to various formats (GeoJSON, CSV)

**Key Features:**
- Automatic data enrichment with NSI attributes (YearBuilt, NumberOfStories, etc.)
- Interactive map visualization
- Multiple export formats for further analysis

## Step 1: Install BrailsPlusPlus

Install the latest version of BrailsPlusPlus from GitHub. This library provides tools for:
- Scraping building footprints from multiple sources (OSM, USA Footprints)
- Integrating with National Structure Inventory (NSI)
- Managing asset inventories

**Note:** This may take a few minutes on first run.

In [ ]:
!pip install --upgrade git+https://github.com/NHERI-SimCenter/BrailsPlusPlus

  Cloning https://github.com/NHERI-SimCenter/BrailsPlusPlus to c:\users\varun\appdata\local\temp\pip-req-build-kxo1uwce
  Resolved https://github.com/NHERI-SimCenter/BrailsPlusPlus to commit 12390ca660e86f13c95faf4bddbbded4f0f01efa
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for brails: filename=brails-4.1.4-py3-none-any.whl size=2317196 sha256=cd2fad26a4e0455776a4974d13314524349b4011fe59d07369816bc8dd2f9866
  Stored in directory: C:\Users\varun\AppData\Local\Temp\pip-ephem-wheel-cache-q6kjw79f\wheels\0d\c3\4e\5d092a6b9f422aabedf42d53704326af544d122b5fcae7fb5e
Successfully built brails
  Attempting uninstall: brails
    Found existing installation: BRAILS 3.1.3
    Uninstalling BRAILS-3.1.

  Running command git clone --filter=blob:none --quiet https://github.com/NHERI-SimCenter/BrailsPlusPlus 'C:\Users\varun\AppData\Local\Temp\pip-req-build-kxo1uwce'


## Step 2: Fetch Building Inventory by Location

Run the inventory script with a location name. The script will:
1. Geocode the location name to coordinates
2. Fetch building footprints from OpenStreetMap
3. Merge with NSI data to add attributes like:
   - YearBuilt
   - NumberOfStories
   - StructureType
   - OccupancyClass
4. Save results to `inventory_<location>_nsi.geojson`

**Alternative:** You can also use a bounding box (see next cell) for more precise area control.

In [21]:
!python get_inventory_simple.py --location "Reseda, CA"

Using location name lookup: Reseda, CA
Fetching footprints with OSM_FootprintScraper ...

Searching for Reseda, CA...
Found Reseda, Los Angeles, Los Angeles County, California, United States

Found a total of 20670 building footprints in Reseda
Total buildings retrieved (footprints): 20670
Merging NSI attributes with footprint inventory ...

Getting National Structure Inventory (NSI) building data for the entered location...
Found a total of 12056 building points in NSI that match the footprint data.
NSI merge complete.
Wrote 20670 assets to c:\Users\varun\OneDrive - Stanford\Desktop\Stanford\5. Autumn 25 Quarter\Independent Study CEE299\Code\data\reseda_loc_nsi.geojson
✅ Wrote 20670 assets to: C:\Users\varun\OneDrive - Stanford\Desktop\Stanford\5. Autumn 25 Quarter\Independent Study CEE299\Code\data\reseda_loc_nsi.geojson
^C


## Alternative: Fetch by Bounding Box

Use this approach when you need precise geographic boundaries.

**Bounding Box Format:** `--bbox LON_MIN LAT_MIN LON_MAX LAT_MAX`
- All coordinates in decimal degrees
- Example: `-118.60 34.20 -118.45 34.30` covers part of Northridge, CA

**Advantages:**
- More precise area control
- Useful for rectangular study areas
- Better for comparing specific regions

Uncomment the cell below to use bounding box mode instead of location name.

In [ ]:
!python get_inventory_simple.py --bbox -118.60 34.20 -118.58 34.32
#--bbox LON_MIN LAT_MIN LON_MAX LAT_MAX

^C


## Step 3: Visualize Buildings on Interactive Map

Create an interactive Folium map to visualize the building footprints.

**What this cell does:**
1. Loads the GeoJSON file containing building inventory
2. Calculates the map center from building centroids
3. Creates a Folium map with buildings overlaid
4. Saves the interactive map as HTML for viewing in a browser

**Output:** 
- Interactive map displayed in notebook
- HTML file saved to `map/reseda_loc_nsi_map.html`

**Note:** You can open the HTML file in any browser to interact with the map (zoom, pan, click features).

In [ ]:
# Import required libraries for mapping and geospatial analysis
import folium
import geopandas as gpd
from shapely.geometry import mapping
import matplotlib.pyplot as plt
import os

# Load the GeoJSON file containing building inventory with NSI attributes
gdf = gpd.read_file("data/reseda_loc_nsi.geojson")

# Calculate the center point of the map using the mean of all building centroids
# This ensures the map is centered on the study area
center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]

# Create a Folium map centered on the study area
# zoom_start=14 provides a good street-level view
m = folium.Map(location=center, zoom_start=14)

# Add the building footprints to the map as a GeoJSON layer
# This will display all buildings with their geometries
folium.GeoJson(gdf).add_to(m)

# Display the map in the notebook
m   

# Create the map directory if it doesn't exist
os.makedirs("map", exist_ok=True)

# Save the interactive map as an HTML file
# You can open this file in any web browser for full interactivity
m.save("map/reseda_loc_nsi_map.html")

C:\Users\varun\AppData\Local\Temp\ipykernel_19048\677157588.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]


## Step 4: Export to CSV Format

Convert the GeoDataFrame to CSV for analysis in spreadsheet software or other tools.

**What gets exported:**
- All building attributes (NSI data)
- Geometry as WKT (Well-Known Text) format
- All numeric and text fields

**Use cases:**
- Statistical analysis in Excel, R, or Python pandas
- Database import
- Sharing data with non-GIS users

**Output:** `csv_data/reseda_loc_nsi.csv`

In [ ]:
# Create the output directory for CSV files if it doesn't exist
os.makedirs("csv_data", exist_ok=True)

# Convert the GeoDataFrame to CSV format
# index=False prevents adding an extra index column
# Geometry will be exported as WKT (Well-Known Text) format
gdf.to_csv("csv_data/reseda_loc_nsi.csv", index=False)

print(f"✅ Exported {len(gdf)} buildings to CSV")
print(f"📁 File saved to: csv_data/reseda_loc_nsi.csv")

## Additional Analysis (Optional)

Use this section to perform additional analysis on the building inventory:

**Suggested analyses:**
- Summary statistics of building attributes
- Year built distribution histograms
- Number of stories analysis
- Occupancy type breakdown
- Spatial clustering analysis

**Example code snippets:**

```python
# View first few rows
print(gdf.head())

# Summary statistics
print(gdf.describe())

# Count by year built decade
if 'YearBuilt' in gdf.columns:
    gdf['Decade'] = (gdf['YearBuilt'] // 10) * 10
    print(gdf['Decade'].value_counts().sort_index())

# Plot building age distribution
if 'YearBuilt' in gdf.columns:
    gdf['YearBuilt'].hist(bins=30)
    plt.xlabel('Year Built')
    plt.ylabel('Count')
    plt.title('Building Age Distribution')
    plt.show()
```